# 딜레이 이펙터

y[n] = x[n] + g * x[n-D]
- D: 딜레이 샘플 수
- g: 딜레이 신호의 게인
- x[n−D]: D만큼 지연된 입력
- x[n]: 원음(Dry)
- 딜레이 샘플 수 = 딜레이사간(ms) * Sample_rate / 1000

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile

class Delay:
    def __init__(self, delay_ms, sample_rate, gain):
        self.delay_ms = delay_ms
        self.sample_rate = sample_rate
        self.gain = gain

    def process(self, x):
        # 딜레이 샘플 수 계산
        D = int(self.delay_ms * self.sample_rate / 1000)
        
        # Numpy 슬라이싱을 이용한 최적화 (파이썬 for문 제거로 속도 대폭 향상)
        # 메아리가 잘리지 않게 원래 길이 + 딜레이 길이만큼 배열 생성
        y = np.zeros(len(x) + D, dtype=np.float32)
        
        # 1. 원래 소리 복사
        y[:len(x)] += x
        
        # 2. 딜레이(메아리) 소리 더하기
        y[D:len(x) + D] += self.gain * x
        
        return y


In [ ]:
# 1. 파일 읽기 및 실수형(float) 변환
sample_rate, data = wavfile.read('sine_440.wav')
if len(data.shape) > 1:
    data = data[:, 0]
data = data.astype(np.float32)

# 2. 원본 파형 그리기 (처음 20ms 구간)
time_original = np.linspace(0., len(data) / sample_rate, len(data))
plt.figure(figsize=(12, 4))
plt.plot(time_original, data)
plt.title('Original Waveform (First 20ms)')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.xlim(0, 0.02)
plt.grid(True)
plt.show()

# 3. Delay 클래스를 이용한 딜레이 적용 (300ms, gain 0.5)
my_delay = Delay(delay_ms=300, sample_rate=sample_rate, gain=0.5)
output_data = my_delay.process(data)

# 4. 정규화 및 오디오 파일 저장
max_amp = np.max(np.abs(output_data))
if max_amp > 0:
    output_data = output_data / max_amp
wavfile.write('sine_440_final_delay.wav', sample_rate, np.int16(output_data * 32767))
print('딜레이가 적용된 파일이 저장되었습니다: sine_440_final_delay.wav')

# 5. 딜레이가 적용된 파형 그리기 (앞부분 1초 구간)
time_delayed = np.linspace(0., len(output_data) / sample_rate, len(output_data))
plt.figure(figsize=(12, 4))
plt.plot(time_delayed, output_data)
plt.title('Waveform with Delay Effect')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.xlim(0, 1.0)
plt.grid(True)
plt.show()
